# Uncertainty-aware articulation after staged MHR fitting

This notebook isolates and expands the distinctive part of the `articulate/` uncertainty-aware preset for one real FAUST shape. The ordinary production stages still run first:

```text
raw mesh → rendered views → SAM 3D Body → RANSAC alignment
         → robust joint triangulation → pose optimization → surface optimization
         → view residuals → z-buffer visibility → weighted fusion + uncertainty
```

The saved optimized per-view MHR parameters are used as the handoff from the expensive staged optimizer. From there, this notebook reconstructs every retained MHR hypothesis, recomputes view weights, checks visibility, expands the fusion equation, and compares the result numerically with the production OBJ and confidence sidecar.

The values mirror `articulate/config/uncertainty_aware.yaml`. No production code is changed and no full-dataset inference is run.

## 1. Setup and explicit assets

In [ ]:
import os

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "articulate").is_dir():
            return candidate
    fallback = Path("/home/knpob/work/code/atm-dev")
    if (fallback / "pyproject.toml").is_file():
        return fallback
    raise RuntimeError("Could not locate the atm-dev repository root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh

from articulate.output_handler.uncertainty_aware_output_handler import (
    UncertaintyAwareOutputHandler,
)
from articulate.shape_prior.mhr_prior import MHRShapePrior
from articulate.shape_refiner.uncertainty_aware_mhr_refiner import (
    _softmax_negative,
    confidence_weighted_fusion,
    vertex_visibility_from_depth,
)
from articulate.view_sampler.multi_view_sampler import MultiViewSampler, render_mesh

# One representative production result. Change SHAPE_NAME only if all three
# uncertainty-aware artifacts exist for the replacement.
SHAPE_NAME = "tr_reg_080"
DATA_ROOT = Path("/home/knpob/work/data/shape-corr/FAUST_r")
ARTICULATION_ROOT = PROJECT_ROOT / "output/faust_r_uncertainty"
RAW_PATH = DATA_ROOT / "off" / f"{SHAPE_NAME}.off"
MHR_DIR = ARTICULATION_ROOT / "mhr"
MHR_OBJ_PATH = MHR_DIR / f"{SHAPE_NAME}.obj"
MHR_PARAMS_PATH = MHR_DIR / f"{SHAPE_NAME}.npy"
CONFIDENCE_PATH = MHR_DIR / f"{SHAPE_NAME}.confidence.npz"
MHR_MODEL_PATH = (
    PROJECT_ROOT
    / "pkg/sam_3d_body/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VIEW_TEMPERATURE = 1.0
VISIBILITY_DEPTH_TOLERANCE = 1e-3
VISIBILITY_PIXEL_RADIUS = 1
FUSION_EPS = 1e-8
DOWNSTREAM_UNCERTAINTY_SCALE = 0.02  # mirrors match/config/map_extractor/partial_aware.yaml
PYVISTA_BACKEND = "static"  # Use "html" for rotatable scenes.

required = [
    RAW_PATH,
    MHR_OBJ_PATH,
    MHR_PARAMS_PATH,
    CONFIDENCE_PATH,
    MHR_MODEL_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, (
    "Missing required assets. Run the uncertainty-aware articulation preset first: "
    + ", ".join(missing)
)

pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

print(f"Shape:      {SHAPE_NAME}")
print(f"Raw mesh:   {RAW_PATH}")
print(f"Artifacts:  {MHR_DIR}")
print(f"Device:     {DEVICE}")

In [ ]:
def load_trimesh(path):
    mesh = trimesh.load(path, process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a triangular mesh at {path}, got {type(mesh)}")
    return mesh

def as_polydata(vertices, faces):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces, dtype=np.int64)
    pv_faces = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    return pv.PolyData(vertices, pv_faces)

def add_scalar_mesh(plotter, vertices, faces, values, title, *, cmap="viridis", clim=None):
    mesh = as_polydata(vertices, faces)
    mesh.point_data[title] = np.asarray(values)
    plotter.add_mesh(
        mesh,
        scalars=title,
        cmap=cmap,
        clim=clim,
        smooth_shading=True,
        show_scalar_bar=True,
    )
    plotter.add_title(title, font_size=10)
    plotter.camera_position = "xy"

def tensor(array):
    return torch.as_tensor(array, dtype=torch.float32, device=DEVICE)

## 2. Load the optimizer handoff and confidence sidecar

The standard staged optimizer stores one MHR parameter set and similarity transform per retained view. The uncertainty-aware output handler leaves that legacy `.npy` schema unchanged and stores confidence separately in an `allow_pickle=False` NPZ. Vertex IDs in the OBJ and sidecar must share the fixed MHR topology.

In [ ]:
raw_mesh = load_trimesh(RAW_PATH)
saved_fused_mesh = load_trimesh(MHR_OBJ_PATH)
params = np.load(MHR_PARAMS_PATH, allow_pickle=True).item()
with np.load(CONFIDENCE_PATH, allow_pickle=False) as archive:
    confidence = {name: np.asarray(archive[name]) for name in archive.files}

required_param_keys = {
    "identity_coeffs", "model_parameters", "face_expr_coeffs", "R", "t", "s"
}
required_confidence_keys = {
    "schema_version",
    "view_keys",
    "residual_names",
    "view_residual_terms",
    "view_residuals",
    "view_weights",
    "vertex_visibility",
    "vertex_visibility_support",
    "vertex_visibility_fallback",
    "vertex_uncertainty",
}
assert required_param_keys <= params.keys()
assert required_confidence_keys <= confidence.keys()
assert int(confidence["schema_version"]) == 1

view_count = len(confidence["view_keys"])
vertex_count = len(saved_fused_mesh.vertices)
assert all(np.asarray(params[key]).shape[0] == view_count for key in required_param_keys)
assert confidence["vertex_visibility"].shape == (view_count, vertex_count)
assert len(confidence["vertex_uncertainty"]) == vertex_count
assert len(confidence["vertex_visibility_support"]) == vertex_count

print(f"Retained views:       {view_count}")
print(f"Shared MHR vertices:  {vertex_count:,}")
print(f"View keys (az, el):   {confidence['view_keys'].tolist()}")
print(f"Sidecar fields:       {sorted(confidence)}")

## 3. Reconstruct and compare the retained view hypotheses

Each retained rendered view produced its own optimized MHR parameter set. Because all hypotheses use the same MHR topology, vertex ID (m) refers to the same anatomy in every view.

This section now uses two separate figures:

1. **Geometry overlays** show the raw target and each optimized hypothesis with plain colors. No heatmap is involved.
2. **View-to-view disagreement maps** color vertex (m) by the Euclidean distance between that view's vertex and the unweighted mean of the same vertex across views. All panels use one shared color scale: dark means the hypotheses agree there; bright means that particular view departs from the others.

The disagreement map is an explanatory diagnostic—not the production confidence or uncertainty. Production confidence additionally uses residuals and visibility in the following sections.

In [ ]:
shape_prior = MHRShapePrior(mhr_path=str(MHR_MODEL_PATH), device=DEVICE)
shape_prior.load()

with torch.no_grad():
    per_view_vertices = shape_prior.generate(
        tensor(params["identity_coeffs"]),
        tensor(params["model_parameters"]),
        tensor(params["face_expr_coeffs"]),
        tensor(params["R"]),
        tensor(params["s"]),
        tensor(params["t"]),
        return_keypoints=False,
    ).detach().cpu().numpy()

assert per_view_vertices.shape == (view_count, vertex_count, 3)
plain_view_mean = per_view_vertices.mean(axis=0)
per_view_disagreement = np.linalg.norm(
    per_view_vertices - plain_view_mean[None, :, :], axis=2
)
view_colors = ["#4c78a8", "#f28e2b", "#59a14f", "#e15759"]

# Figure 1: geometry only. These are fitted 3D hypotheses, not camera images.
plotter = pv.Plotter(shape=(1, view_count + 1), window_size=(1900, 520))
plotter.subplot(0, 0)
plotter.add_mesh(
    pv.wrap(raw_mesh), color="lightgray", opacity=0.55, label="raw target",
    smooth_shading=True,
)
plotter.add_mesh(
    pv.wrap(saved_fused_mesh), color="tomato", opacity=0.65, label="saved fusion",
    smooth_shading=True,
)
plotter.add_title("reference: raw + fused MHR", font_size=10)
plotter.camera_position = "xy"

for index, view_key in enumerate(confidence["view_keys"]):
    plotter.subplot(0, index + 1)
    plotter.add_mesh(
        pv.wrap(raw_mesh), color="lightgray", opacity=0.22, smooth_shading=True,
    )
    plotter.add_mesh(
        as_polydata(per_view_vertices[index], saved_fused_mesh.faces),
        color=view_colors[index % len(view_colors)],
        opacity=0.82,
        smooth_shading=True,
    )
    plotter.add_title(
        f"hypothesis from rendered view\naz={view_key[0]:g}°, el={view_key[1]:g}°",
        font_size=10,
    )
    plotter.camera_position = "xy"

plotter.link_views()
plotter.show()

# Figure 2: one shared scalar and one shared scale across the retained views.
disagreement_clim = [
    0.0,
    float(np.quantile(per_view_disagreement, 0.99)),
]
plotter = pv.Plotter(shape=(1, view_count), window_size=(1650, 560))
for index, view_key in enumerate(confidence["view_keys"]):
    plotter.subplot(0, index)
    mesh = as_polydata(per_view_vertices[index], saved_fused_mesh.faces)
    scalar_name = "distance from unweighted view mean"
    mesh.point_data[scalar_name] = per_view_disagreement[index]
    plotter.add_mesh(
        mesh,
        scalars=scalar_name,
        cmap="magma",
        clim=disagreement_clim,
        smooth_shading=True,
        show_scalar_bar=(index == view_count - 1),
        scalar_bar_args={"title": "distance\n(mesh units)"},
    )
    plotter.add_title(
        f"view {tuple(view_key)}\n"
        f"mean={per_view_disagreement[index].mean():.2e}, "
        f"p95={np.quantile(per_view_disagreement[index], 0.95):.2e}",
        font_size=9,
    )
    plotter.camera_position = "xy"

plotter.link_views()
plotter.show()

## 4. Residual heatmap and per-view confidence

For retained view (v), production stores diagnostic terms in the order

$$
(\widetilde E_v^{\mathrm{align}},\widetilde E_v^{\mathrm{surf}},\widetilde E_v^{\mathrm{kp}},
 1-\mathrm{IoU}_v,1-\mathrm{InlierRatio}_v).
$$

The heatmap below makes the full view-by-term table explicit. Alignment is shown for diagnosis but is not included again in the confidence residual. With the preset's unit coefficients,

$$
r_v=\widetilde E_v^{\mathrm{surf}}+\widetilde E_v^{\mathrm{kp}}
 +(1-\mathrm{IoU}_v)+(1-\mathrm{InlierRatio}_v),\qquad
w_v=\frac{e^{-r_v/\tau}}{\sum_{v'}e^{-r_{v'}/\tau}}.
$$

A smaller residual therefore produces a larger global view weight. The weight is one scalar per retained view; per-vertex visibility is applied only in the next section.

In [ ]:
residual_names = confidence["residual_names"].tolist()
terms = confidence["view_residual_terms"].astype(np.float64)
stored_residuals = confidence["view_residuals"].astype(np.float64)
stored_weights = confidence["view_weights"].astype(np.float64)

# Expand the configured residual equation. Column 0 (alignment) is diagnostic.
manual_residuals = terms[:, 1] + terms[:, 2] + terms[:, 3] + terms[:, 4]
logits = -manual_residuals / VIEW_TEMPERATURE
logits -= logits.max()
manual_weights = np.exp(logits)
manual_weights /= manual_weights.sum()
helper_weights = _softmax_negative(manual_residuals, VIEW_TEMPERATURE)

np.testing.assert_allclose(manual_residuals, stored_residuals, atol=1e-12)
np.testing.assert_allclose(manual_weights, helper_weights, atol=1e-12)
np.testing.assert_allclose(manual_weights, stored_weights, atol=1e-12)
np.testing.assert_allclose(stored_weights.sum(), 1.0, atol=1e-12)

labels = [f"{az:g}°, {el:g}°" for az, el in confidence["view_keys"]]
term_labels = ["alignment\n(diagnostic)", "surface", "keypoint", "1 − IoU", "1 − inlier"]
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)

# View-by-residual heatmap with the numerical values printed in every cell.
heatmap_max = float(terms.max())
image = axes[0, 0].imshow(
    terms, cmap="magma", vmin=0.0, vmax=heatmap_max, aspect="auto"
)
axes[0, 0].set_xticks(np.arange(len(term_labels)), term_labels)
axes[0, 0].set_yticks(np.arange(view_count), labels)
axes[0, 0].set_xlabel("stored residual term")
axes[0, 0].set_ylabel("retained view (azimuth, elevation)")
axes[0, 0].set_title("Residual heatmap: darker is better")
for row in range(view_count):
    for column in range(terms.shape[1]):
        axes[0, 0].text(
            column,
            row,
            f"{terms[row, column]:.3f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if terms[row, column] > heatmap_max / 2 else "black",
        )
fig.colorbar(image, ax=axes[0, 0], label="normalized penalty")

# Only these four terms contribute to r_v.
bottom = np.zeros(view_count)
for column, name in zip(range(1, 5), term_labels[1:]):
    axes[0, 1].bar(labels, terms[:, column], bottom=bottom, label=name)
    bottom += terms[:, column]
axes[0, 1].plot(labels, stored_residuals, "ko", label="sum = residual")
axes[0, 1].set_title("Terms that actually form the confidence residual")
axes[0, 1].set_ylabel("normalized penalty")
axes[0, 1].legend(fontsize=8)

bar_colors = plt.cm.viridis(stored_weights / stored_weights.max())
bars = axes[1, 0].bar(labels, stored_weights, color=bar_colors)
axes[1, 0].bar_label(bars, labels=[f"w={w:.3f}" for w in stored_weights], padding=3)
for position, residual in enumerate(stored_residuals):
    axes[1, 0].text(
        position, 0.01, f"r={residual:.3f}", ha="center", va="bottom",
        color="white", fontsize=8, rotation=90,
    )
axes[1, 0].set_ylim(0, max(stored_weights) * 1.25)
axes[1, 0].set_title(f"Global per-view confidence (temperature={VIEW_TEMPERATURE:g})")
axes[1, 0].set_ylabel("softmax mass")

temperatures = [0.1, 0.25, 0.5, 1.0, 2.0]
for temperature in temperatures:
    axes[1, 1].plot(
        np.arange(view_count),
        _softmax_negative(stored_residuals, temperature),
        marker="o",
        label=f"τ={temperature:g}",
    )
axes[1, 1].set_xticks(np.arange(view_count), labels)
axes[1, 1].set_title("Temperature controls selectivity")
axes[1, 1].set_ylabel("view weight")
axes[1, 1].legend(fontsize=8)
plt.show()

### Per-view confidence on the fitted hypotheses

Each mesh below is colored uniformly because (w_v) is a **global view scalar**, not a vertex-wise quantity. The shared color scale and titles connect the residual heatmap to the geometry: lower (r_v) means higher (w_v). Section 5 will gate this scalar by per-vertex visibility.

In [ ]:
plotter = pv.Plotter(shape=(1, view_count), window_size=(1650, 560))
for index, view_key in enumerate(confidence["view_keys"]):
    plotter.subplot(0, index)
    mesh = as_polydata(per_view_vertices[index], saved_fused_mesh.faces)
    mesh.point_data["global view weight"] = np.full(vertex_count, stored_weights[index])
    plotter.add_mesh(
        mesh,
        scalars="global view weight",
        cmap="viridis",
        clim=[0.0, float(stored_weights.max())],
        smooth_shading=True,
        show_scalar_bar=(index == view_count - 1),
        scalar_bar_args={"title": "global\nview weight"},
    )
    plotter.add_title(
        f"view {tuple(view_key)}\nr={stored_residuals[index]:.3f}, "
        f"w={stored_weights[index]:.3f}",
        font_size=10,
    )
    plotter.camera_position = "xy"
plotter.link_views()
plotter.show()

## 5. Per-view confidence becomes per-vertex fusion weight through visibility

A good global view is not evidence for every vertex. Production rerenders each optimized MHR in its original camera and tests each projected vertex against the z-buffer using

$$
u=fX/Z+c_x,\qquad v=c_y-fY/Z.
$$

The minus sign is essential because camera (Y) points upward while image rows increase downward. The binary mask (
u_{v,m}) turns the global view confidence into a per-view, per-vertex contribution (w_v
u_{v,m}): an occluded vertex gets zero contribution even when the view is globally reliable.

Rendering can differ by a few boundary pixels across graphics drivers, so the regenerated mask is checked for high agreement; exact fusion equivalence below uses the stored production mask.

In [ ]:
view_sampler = MultiViewSampler(
    elevation_config="multi_3level",
    rotation_step=45.0,
    camera_dist=2.0,
    axis_order="xyz",
    camera_configs={
        "focal_length": 500.0,
        "camera_center": (256, 256),
        "h": 512,
        "w": 512,
    },
)

body_diagonal = float(np.linalg.norm(np.ptp(raw_mesh.bounds, axis=0)))
absolute_depth_tolerance = VISIBILITY_DEPTH_TOLERANCE * body_diagonal
regenerated_visibility = []

for vertices, view_key in zip(per_view_vertices, confidence["view_keys"]):
    # render(raw_mesh, key) reconstructs the exact input-camera convention.
    camera = view_sampler.render(raw_mesh, tuple(view_key))["cam"]
    hypothesis = trimesh.Trimesh(
        vertices=vertices, faces=saved_fused_mesh.faces, process=False
    )
    _, depth = render_mesh(
        hypothesis,
        camera_pose=camera["cam_pose"],
        **camera["intrinsics"],
    )
    regenerated_visibility.append(
        vertex_visibility_from_depth(
            vertices,
            depth,
            camera,
            depth_tolerance=absolute_depth_tolerance,
            pixel_radius=VISIBILITY_PIXEL_RADIUS,
        )
    )

regenerated_visibility = np.stack(regenerated_visibility)
stored_visibility = confidence["vertex_visibility"].astype(bool)
visibility_agreement = float(np.mean(regenerated_visibility == stored_visibility))
assert visibility_agreement > 0.99, (
    f"Unexpected visibility disagreement: agreement={visibility_agreement:.4f}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = np.arange(view_count)
axes[0].bar(x - 0.18, stored_visibility.sum(axis=1), width=0.36, label="stored")
axes[0].bar(x + 0.18, regenerated_visibility.sum(axis=1), width=0.36, label="regenerated")
axes[0].set_xticks(x, labels)
axes[0].set_ylabel("visible MHR vertices")
axes[0].set_title(f"Visibility masks: {100 * visibility_agreement:.2f}% agreement")
axes[0].legend()

axes[1].hist(
    confidence["vertex_visibility_support"], bins=50, color="#59a14f", alpha=0.85
)
axes[1].set_xlabel("weighted visibility support")
axes[1].set_ylabel("vertex count")
axes[1].set_title("Support is not the same as low variance")
plt.show()

# Show the actual per-view, per-vertex weight entering fusion.
view_vertex_contribution = stored_weights[:, None] * stored_visibility.astype(np.float64)
plotter = pv.Plotter(shape=(1, view_count), window_size=(1650, 560))
for index, view_key in enumerate(confidence["view_keys"]):
    plotter.subplot(0, index)
    mesh = as_polydata(per_view_vertices[index], saved_fused_mesh.faces)
    scalar_name = "fusion contribution w_v × visibility"
    mesh.point_data[scalar_name] = view_vertex_contribution[index]
    plotter.add_mesh(
        mesh,
        scalars=scalar_name,
        cmap="viridis",
        clim=[0.0, float(stored_weights.max())],
        smooth_shading=True,
        show_scalar_bar=(index == view_count - 1),
        scalar_bar_args={"title": "per-vertex\nfusion weight"},
    )
    plotter.add_title(
        f"view {tuple(view_key)}\ndark=occluded, bright=w={stored_weights[index]:.3f}",
        font_size=9,
    )
    plotter.camera_position = "xy"
plotter.link_views()
plotter.show()

## 6. Fusion, uncertainty, and per-vertex confidence heatmaps

For canonical vertex (m), the effective weight is (w_{v,m}=w_v
u_{v,m}). The stored weighted visibility support is

$$
s_m=\sum_v w_v\nu_{v,m}.
$$

Unseen vertices fall back to the global view weights so the output remains a valid mesh, but their stored support remains zero. Production also stores cross-view consensus variance (sigma_m^2). A low variance is only reassuring when support is also strong.

To provide the requested per-vertex confidence illustration, the last visualization derives

$$
q_m^{\mathrm{art}} =
s_m\exp\!\left(-\frac{\sigma_m^2}{2s_u^2}\right),
\qquad s_u=0.02\times\text{body diagonal}.
$$

This derived factor mirrors the visibility and uncertainty factors used by the downstream partial-aware matcher. It is **not** a stored output or a calibrated probability, and the final raw-point confidence still includes proxy-distance and normal-agreement factors.

In [ ]:
# Manual expansion of confidence_weighted_fusion().
per_vertex_weights = stored_weights[:, None] * stored_visibility.astype(np.float64)
manual_support = per_vertex_weights.sum(axis=0)
manual_fallback = manual_support <= FUSION_EPS
per_vertex_weights[:, manual_fallback] = stored_weights[:, None]

denominator = per_vertex_weights.sum(axis=0)
manual_fused = (
    per_vertex_weights[:, :, None] * per_view_vertices
).sum(axis=0) / np.maximum(denominator[:, None], FUSION_EPS)
squared_deviation = np.sum(
    (per_view_vertices - manual_fused[None, :, :]) ** 2, axis=2
)
manual_uncertainty = (
    per_vertex_weights * squared_deviation
).sum(axis=0) / np.maximum(denominator, FUSION_EPS)

helper_fused, helper_uncertainty, helper_support, helper_fallback = (
    confidence_weighted_fusion(
        per_view_vertices,
        stored_weights,
        stored_visibility,
        eps=FUSION_EPS,
    )
)

np.testing.assert_allclose(manual_fused, helper_fused, atol=1e-12)
np.testing.assert_allclose(manual_uncertainty, helper_uncertainty, atol=1e-12)
np.testing.assert_allclose(manual_support, helper_support, atol=1e-12)
np.testing.assert_array_equal(manual_fallback, helper_fallback)

# Compare with the production artifacts written by merge_meshes() and the output handler.
np.testing.assert_allclose(manual_fused, saved_fused_mesh.vertices, atol=1e-6)
np.testing.assert_allclose(
    manual_uncertainty, confidence["vertex_uncertainty"], atol=1e-8
)
np.testing.assert_allclose(
    manual_support, confidence["vertex_visibility_support"], atol=1e-12
)
np.testing.assert_array_equal(
    manual_fallback, confidence["vertex_visibility_fallback"].astype(bool)
)

print("Expanded fusion matches confidence_weighted_fusion() and the saved OBJ.")
print(f"Fallback vertices (no retained view saw them): {manual_fallback.sum():,}")
print(
    "Uncertainty range: "
    f"[{manual_uncertainty.min():.3e}, {manual_uncertainty.max():.3e}]"
)

In [ ]:
uncertainty_log = np.log10(manual_uncertainty + 1e-10)
absolute_uncertainty_scale = DOWNSTREAM_UNCERTAINTY_SCALE * body_diagonal
inverse_uncertainty_factor = np.exp(
    -manual_uncertainty / (2.0 * absolute_uncertainty_scale**2)
)
derived_articulation_confidence = (
    np.clip(manual_support, 0.0, 1.0) * inverse_uncertainty_factor
)

plotter = pv.Plotter(shape=(1, 5), window_size=(2400, 520))
panels = [
    (
        manual_support,
        "1. stored support s_m\nhigher = better observed",
        "viridis",
        [0.0, 1.0],
    ),
    (
        uncertainty_log,
        "2. stored log10 variance\nhigher = worse agreement",
        "magma",
        [
            float(np.quantile(uncertainty_log, 0.01)),
            float(np.quantile(uncertainty_log, 0.99)),
        ],
    ),
    (
        inverse_uncertainty_factor,
        "3. inverse-uncertainty factor\nhigher = better agreement",
        "viridis",
        [0.0, 1.0],
    ),
    (
        derived_articulation_confidence,
        "4. derived articulation factor\nsupport × agreement",
        "viridis",
        [0.0, 1.0],
    ),
    (
        manual_fallback.astype(float),
        "5. stored fallback mask\n1 = no view observed",
        "coolwarm",
        [0.0, 1.0],
    ),
]
for column, (values, title, cmap, clim) in enumerate(panels):
    plotter.subplot(0, column)
    add_scalar_mesh(
        plotter,
        manual_fused,
        saved_fused_mesh.faces,
        values,
        title,
        cmap=cmap,
        clim=clim,
    )
plotter.link_views()
plotter.show()

print(
    "Derived articulation confidence factor range: "
    f"[{derived_articulation_confidence.min():.3f}, "
    f"{derived_articulation_confidence.max():.3f}]"
)

## 7. The sidecar stays separate and versioned

This temporary round trip exercises the production output handler without writing notebook outputs into the repository. The ordinary MHR parameter file remains compatible with legacy consumers, while matching can opt into the confidence NPZ.

In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    handler = UncertaintyAwareOutputHandler(out_dir=temp_dir)
    roundtrip_path = handler.store_confidence(SHAPE_NAME, confidence)
    with np.load(roundtrip_path, allow_pickle=False) as roundtrip:
        assert set(roundtrip.files) == set(confidence)
        for name in confidence:
            np.testing.assert_array_equal(roundtrip[name], confidence[name])

print("Confidence sidecar round trip succeeded with allow_pickle=False.")

In [ ]:
diagnostics = {
    "shape": SHAPE_NAME,
    "retained_views": view_count,
    "canonical_vertices": vertex_count,
    "view_weights_sum": float(stored_weights.sum()),
    "visibility_regeneration_agreement": visibility_agreement,
    "visible_vertices_per_view": stored_visibility.sum(axis=1).tolist(),
    "zero_support_fallback_vertices": int(manual_fallback.sum()),
    "derived_articulation_confidence_range": [
        float(derived_articulation_confidence.min()),
        float(derived_articulation_confidence.max()),
    ],
    "max_saved_obj_fusion_error": float(
        np.max(np.abs(manual_fused - saved_fused_mesh.vertices))
    ),
    "max_saved_uncertainty_error": float(
        np.max(np.abs(manual_uncertainty - confidence["vertex_uncertainty"]))
    ),
}
diagnostics

## Visual correctness checklist

- In Section 3, the first row is geometry only; the second row shows distance from the unweighted per-vertex view mean on one shared scale.
- Bright disagreement in one Section 3 panel means that view's hypothesis departs locally from the other hypotheses. It is not yet the production uncertainty.
- In the residual heatmap, smaller cells are better. The alignment column is diagnostic; the other four columns sum to the plotted residual.
- Lower residual views must receive larger global softmax weights, and the weights must sum to one.
- A per-view confidence mesh is uniformly colored because (w_v) is global; the visibility-gated maps should contain both zero and (w_v).
- Regenerated z-buffer masks should agree with the stored masks except for a small number of renderer-dependent silhouette pixels.
- Visibility should change with viewpoint. A mask that marks nearly every vertex visible usually indicates a projection or depth-convention error.
- The stored support and stored variance maps must be interpreted together. Low variance with weak support is not high confidence.
- The derived articulation factor is only an illustration of the two downstream articulation terms, not a calibrated probability.
- Zero-support vertices may still have valid fused coordinates, but they must be marked as fallback/prior-filled.
- The manually expanded fusion must match the production helper, saved OBJ, and saved uncertainty arrays within the asserted tolerances.